### **SCHEMA DESIGN**  

Vertically stack all the traits into one dataset.  
   • Skills  
   • Abilities  
   • Work Styles 


**BUILD FINAL TABLE FOR PROMPT AND KG DESIGN**   
Do left join to link occupations to all traits 


In [16]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add it to the system path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [18]:

import pandas as pd
from pathlib import Path
from src.utils.functions import find_project_root

In [ ]:
PROJECT_ROOT = find_project_root()
(PROJECT_ROOT / "data").exists()

True

In [ ]:
# SET UP PATHS TO FILES AND DIRECTORIES

#--- DIRECTORIES ---#
raw_dir = PROJECT_ROOT / "data/onet_datasets/raw"
output_dir = PROJECT_ROOT / "data/onet_datasets/curated"

#---- FILES ---#
occupation_file = raw_dir / "Occupation_Data.txt"
skills_file = raw_dir / "Skills.txt"
abilities_file = raw_dir / "Abilities.txt"
work_styles_file = raw_dir / "Work_Styles.txt"

datasets = [occupation_file, abilities_file, skills_file, work_styles_file]
print(datasets)

[PosixPath('/Users/f.kissi/Documents/RAV/data/onet_datasets/raw/Occupation_Data.txt'), PosixPath('/Users/f.kissi/Documents/RAV/data/onet_datasets/raw/Abilities.txt'), PosixPath('/Users/f.kissi/Documents/RAV/data/onet_datasets/raw/Skills.txt'), PosixPath('/Users/f.kissi/Documents/RAV/data/onet_datasets/raw/Work_Styles.txt')]


In [40]:
#--- LOADING DATASETS ---#
def load_datasets(file_paths):
    dataframes = {}
    for file_path in file_paths:
        # .stem gets file names automatically
        df_name = file_path.stem 
    
        # pd.read_csv accepts Path objects directly, so no conversion needed here
        df = pd.read_csv(file_path, sep="\t", low_memory=False)
        
        dataframes[df_name] = df
    return dataframes

dataframes = load_datasets(datasets)
print("Datasets loaded successfully.",dataframes.keys())

Datasets loaded successfully. dict_keys(['Occupation_Data', 'Abilities', 'Skills', 'Work_Styles'])


In [41]:
#---- CLEAN DATASETS ----#
def clean_column_names(df):
    df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_").str.replace("*","")
    return df

#---- CLEANING ALL DATAFRAMES ----#
for name in dataframes:
    dataframes[name] = clean_column_names(dataframes[name])

In [42]:
#--CHECK COLUMN NAMES --#
for df_name in dataframes:
    temp_df = dataframes[df_name]
    if "Scale_ID" in temp_df.columns:
        unique_scales = temp_df['Scale_ID'].unique()
        print(f"{df_name}: Unique Scale_IDs: {unique_scales}")
    row_count = len(temp_df)
    print(f"{df_name}: Columns: {temp_df.columns.tolist()} | Total Rows: {row_count}")

Occupation_Data: Columns: ['ONET_SOC_Code', 'Title', 'Description'] | Total Rows: 1016
Abilities: Unique Scale_IDs: ['IM' 'LV']
Abilities: Columns: ['ONET_SOC_Code', 'Element_ID', 'Element_Name', 'Scale_ID', 'Data_Value', 'N', 'Standard_Error', 'Lower_CI_Bound', 'Upper_CI_Bound', 'Recommend_Suppress', 'Not_Relevant', 'Date', 'Domain_Source'] | Total Rows: 92976
Skills: Unique Scale_IDs: ['IM' 'LV']
Skills: Columns: ['ONET_SOC_Code', 'Element_ID', 'Element_Name', 'Scale_ID', 'Data_Value', 'N', 'Standard_Error', 'Lower_CI_Bound', 'Upper_CI_Bound', 'Recommend_Suppress', 'Not_Relevant', 'Date', 'Domain_Source'] | Total Rows: 62580
Work_Styles: Unique Scale_IDs: ['DR' 'WI']
Work_Styles: Columns: ['ONET_SOC_Code', 'Element_ID', 'Element_Name', 'Scale_ID', 'Data_Value', 'Date', 'Domain_Source'] | Total Rows: 37422


In [43]:
#---- FILTER DATASETS ----#

def filter_dataset(df, relevant_column,column_value):
    filtered_df = df[df[relevant_column] == column_value]
    return filtered_df

#----- APPLY FILTERS -----#
abilites_df = filter_dataset(dataframes["Abilities"], "Scale_ID", "IM").copy()
skills_df = filter_dataset(dataframes["Skills"], "Scale_ID", "IM").copy()
work_styles_df = filter_dataset(dataframes["Work_Styles"], "Scale_ID", "WI").copy()
 

cleaned_dfs = {
    "Abilities": abilites_df,
    "Skills": skills_df,
    "Work_Styles": work_styles_df,
}

In [44]:
#--CHECK COLUMN NAMES --#
for df in dataframes:
    temp_df = dataframes[df]
    print(temp_df.columns.tolist())

['ONET_SOC_Code', 'Title', 'Description']
['ONET_SOC_Code', 'Element_ID', 'Element_Name', 'Scale_ID', 'Data_Value', 'N', 'Standard_Error', 'Lower_CI_Bound', 'Upper_CI_Bound', 'Recommend_Suppress', 'Not_Relevant', 'Date', 'Domain_Source']
['ONET_SOC_Code', 'Element_ID', 'Element_Name', 'Scale_ID', 'Data_Value', 'N', 'Standard_Error', 'Lower_CI_Bound', 'Upper_CI_Bound', 'Recommend_Suppress', 'Not_Relevant', 'Date', 'Domain_Source']
['ONET_SOC_Code', 'Element_ID', 'Element_Name', 'Scale_ID', 'Data_Value', 'Date', 'Domain_Source']


In [45]:
for name, df in cleaned_dfs.items():
    if 'Scale_ID' in df.columns:
        unique_scales = df['Scale_ID'].unique()
        row_count = len(df)
        print(f"--- Dataset: {name} ---")
        print(f"Unique Scales: {unique_scales}")
        print(f"Total Rows: {row_count}")
        
        # Quick logic check
        if len(unique_scales) > 1:
            print("⚠️ WARNING: More than one scale detected!")
        else:
            print("✅ Filter looks good.")
    print("\n")

--- Dataset: Abilities ---
Unique Scales: ['IM']
Total Rows: 46488
✅ Filter looks good.


--- Dataset: Skills ---
Unique Scales: ['IM']
Total Rows: 31290
✅ Filter looks good.


--- Dataset: Work_Styles ---
Unique Scales: ['WI']
Total Rows: 18711
✅ Filter looks good.




In [46]:
#--- VERTICALLY STACK DATASETS ---#

def stack(dataframes_dict):
    stacked_list = []
    
    for name, df in dataframes_dict.items():
        #Add the 'Source' label (e.g., 'Abilities')
        df['Source_Dataset'] = name
        
        # Keep only the columns that exist in all files to ensure a clean stack
        # Standard O*NET columns found in all three:
        common_cols = ['ONET_SOC_Code','Element_Name', 'Data_Value', 'Source_Dataset']
        
        # Only take columns that actually exist in this specific DF
        existing_cols = [c for c in common_cols if c in df.columns]
        stacked_list.append(df[existing_cols])
    
    # The Vertical Join (Stacking)
    combined_df = pd.concat(stacked_list, axis=0, ignore_index=True)
    return combined_df

#--- STACKING ALL CLEANED DATAFRAMES ---#
stacked_df = stack(cleaned_dfs)
print(stacked_df.head())
print(f"Final stacked dataset shape: {stacked_df.shape}")

  ONET_SOC_Code           Element_Name  Data_Value Source_Dataset
0    11-1011.00     Oral Comprehension        4.62      Abilities
1    11-1011.00  Written Comprehension        4.25      Abilities
2    11-1011.00        Oral Expression        4.50      Abilities
3    11-1011.00     Written Expression        4.12      Abilities
4    11-1011.00       Fluency of Ideas        3.88      Abilities
Final stacked dataset shape: (96489, 4)


In [47]:

#--- IDENTIFY ROLES WITH ALL THREE TRAIT TYPES ---#
coverage = (
    stacked_df
      .groupby(["ONET_SOC_Code", "Source_Dataset"])
      .size()
      .unstack(fill_value=0)
)

eligible_roles = coverage[
    (coverage["Abilities"] > 0) &
    (coverage["Skills"] > 0) &
    (coverage["Work_Styles"] > 0)
].index


In [48]:

#--- PREPARING FOR JOINING ---#
# Identify main dataset and others
main_dataset_name = "Occupation_Data"
main_df = dataframes[main_dataset_name]

#--- FILTER DATA FOR ELIGIBLE ROLES ---#
stacked_df = stacked_df[
    stacked_df["ONET_SOC_Code"].isin(eligible_roles)
].copy()

occupation_df = dataframes["Occupation_Data"].loc[
    dataframes["Occupation_Data"]["ONET_SOC_Code"].isin(eligible_roles),
    ["ONET_SOC_Code", "Title", "Description"]
].copy()
print("Occupation Data filtered shape:",occupation_df.shape)

#--- JOINING DATASETS ---#
# Join the metadata (Titles/Descriptions) to the big stacked list but of only eligible roles

final_dataset = pd.merge(
    stacked_df, 
    occupation_df[['ONET_SOC_Code', 'Title']], 
    on='ONET_SOC_Code', 
    how='left'
)

print("Final dataset after merging Titles:",final_dataset.shape)
# T stands for Transpose. This is the "Gold Standard" for checking joins.
print(final_dataset.head(1).T,"\n")

# Count how many rows failed to find a match
print(f"Missing Titles: {final_dataset['Title'].isna().sum()} out of {len(final_dataset)} rows.")

Occupation Data filtered shape: (862, 3)
Final dataset after merging Titles: (93096, 5)
                                 0
ONET_SOC_Code           11-1011.00
Element_Name    Oral Comprehension
Data_Value                    4.62
Source_Dataset           Abilities
Title             Chief Executives 

Missing Titles: 0 out of 93096 rows.


In [49]:
#---- FINAL DATA CLEANING ----#

# Reorder columns for better readability
final_dataset = final_dataset[['ONET_SOC_Code', 'Title', 'Element_Name', 'Data_Value', 'Source_Dataset']]

#---- RENAMING COLUMNS ----#
final_dataset.rename(columns={
    'ONET_SOC_Code': 'Job_Code',
    'Title': 'Job_Title',
    'Element_Name': 'Attribute_Name',
    'Data_Value': 'Importance_Score',
    'Source_Dataset': 'Trait_Type'
}, inplace=True)

In [50]:
#--- SAVING FINAL DATASET ---#
output_file = output_dir / "onet_curated_dataset.csv"
final_dataset.to_csv(output_file, index=False)
print(f"Final dataset saved to: {output_file}")

Final dataset saved to: /Users/f.kissi/Documents/RAV/data/onet_datasets/filtered/onet_curated_dataset.csv
